# Phase 0：认识项目、环境和完整链路

这一课不急着写 RAG 算法。先建立一张工程地图：每个目录解决什么问题，每个阶段的输入和输出是什么，为什么这些阶段必须按顺序连接。

## 本课交付

- 确认 Jupyter 使用的是 `ai-rag-internship` 环境。
- 找到项目根目录和四个阶段模块。
- 看懂一条从原始 Markdown 到 API 引用的完整数据流。
- 生成 `data/processed/project_orientation.json` 作为学习起点记录。

**默认基础：** 会一点 Python 基础，但不了解机器学习和 RAG。每个代码单元格都只解决一个小问题。

## Evidence Quest 任务卡：Mission Control：接手第一宗案件

**你的身份：** 知识库调查员实习生  
**案件背景：** 案件资料散落在 Markdown、PDF 和实验记录里。你的第一项任务是建立调查地图，知道每条证据从哪里来、要交给谁。

### 本关专业 Goal

建立一份可复现的项目任务档案，并说清四阶段如何组成一条产品链路。

### 你要交付的作品

**项目任务地图 + 第一份案件档案**

### 通关判定

- 先运行带逐行中文注释的示范，预测输出，再自己重新敲一遍关键代码。
- 至少改变一个参数或输入，记录它为什么改变了结果。
- 完成末尾的 Boss Challenge，并能解释一个失败样本。
- 把本关产物交给下一关，而不是把代码停留在 Notebook 屏幕上。

**通关奖励：** 解锁徽章：案件接收员  
**学习节奏：** 看故事 -> 跟敲一小段 -> 观察输出 -> 自己改写 -> 验收作品。

## 1. 为什么先认识项目？

RAG 不是一个单独的函数，而是一条流水线。如果不知道流水线的边界，遇到错误时就会把所有问题都归因于“模型不够好”。本项目故意拆成四层：

```text
Phase 1  文件 -> 可追溯 Chunk
Phase 2  Chunk -> 排名结果
Phase 3  排名结果 -> 质量/性能证据
Phase 4  证据 -> 可调用的 API 产品
```

以后每个 Notebook 都会回答三个问题：输入是什么、这一步改变了什么、输出交给谁。

In [1]:
# 导入 Path，用它表示跨平台的文件路径。
from pathlib import Path

# 导入 json，用它读取和保存项目的结构化数据。
import json

# 导入 sys，用它把项目根目录加入 Python 的模块搜索路径。
import sys


# 定义一个函数，负责从当前工作目录向上查找项目根目录。
def find_project_root() -> Path:
    # 把当前目录和它的所有父目录放进候选列表。
    candidates = [Path.cwd(), *Path.cwd().parents]

    # 逐个检查候选目录是否包含本项目的两个核心模块目录。
    for candidate in candidates:
        # 找到同时存在的目录时，返回这个候选目录。
        if (candidate / "phase1_doc_parser").is_dir() and (candidate / "phase2_semantic_search").is_dir():
            return candidate

    # 如果所有候选目录都不符合，说明 Jupyter 启动位置不在项目内。
    raise RuntimeError("找不到项目根目录，请从 ai-search-rag-internship 启动 JupyterLab")


# 执行查找函数，得到当前项目根目录。
ROOT = find_project_root()

# 如果项目根目录还不在模块搜索路径中，就把它添加进去。
if str(ROOT) not in sys.path:
    # 把项目根目录插入最前面，确保导入的是当前项目代码。
    sys.path.insert(0, str(ROOT))

# 打印根目录，帮助学习者确认 Notebook 没有在错误目录运行。
print("项目根目录:", ROOT)

项目根目录: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship


In [2]:
# 定义本关任务编号，后面的记录会用它区分不同阶段。
QUEST_STAGE = 'phase0'

# 定义学习者可以持续保存的案件档案路径。
QUEST_PROFILE_PATH = ROOT / "data" / "processed" / "evidence_quest_profile.json"

# 如果第一次打开课程还没有档案，就使用一个安全的默认案件。
default_profile = {
    "case_name": "校园知识库失踪案",
    "audience": "需要快速查证资料的同学",
    "must_answer": "证据来自哪里，能否回到原文？",
    "must_refuse": "检索结果没有证据时必须说不知道",
    "xp": 0,
    "badges": [],
}

# 检查任务档案是否已经由 Mission Control 创建。
if QUEST_PROFILE_PATH.is_file():
    # 读取学员自己的案件主题，让所有 Notebook 共享同一个故事。
    quest_profile = json.loads(QUEST_PROFILE_PATH.read_text(encoding="utf-8"))
else:
    # 没有档案时复制默认值，避免直接修改模板字典。
    quest_profile = dict(default_profile)

# 计算当前累计经验值；错误值按 0 处理，避免看板阻塞学习。
quest_xp = int(quest_profile.get("xp", 0))

# 读取已经获得的徽章，并复制成当前 Notebook 的列表。
quest_badges = list(quest_profile.get("badges", []))

# 用可见的文字看板告诉学习者自己正在解决哪个真实问题。
print("Evidence Quest / 当前关卡:", QUEST_STAGE)
print("案件:", quest_profile.get("case_name", default_profile["case_name"]))
print("服务对象:", quest_profile.get("audience", default_profile["audience"]))
print("累计 XP:", quest_xp, "| 徽章:", ", ".join(quest_badges) if quest_badges else "尚未获得")

Evidence Quest / 当前关卡: phase0
案件: 校园知识库失踪案
服务对象: 需要复习课程资料的同学
累计 XP: 0 | 徽章: 案件接收员


### 逐句说明这段启动代码

- `Path` 让路径拼接不依赖 Windows 或 Linux 的斜杠写法。
- `json` 让我们保存可重复读取的实验数据，而不是只看屏幕输出。
- `sys.path` 决定 Python 能否导入项目中的 `phase1_doc_parser` 等模块。
- `find_project_root()` 用项目目录特征定位根目录，避免 Notebook 从不同位置启动就失效。

In [3]:
# 定义本项目中必须存在的四个阶段目录。
phase_directories = [
    "phase1_doc_parser",
    "phase2_semantic_search",
    "phase3_optimization_eval",
    "phase4_mini_rag_system",
]

# 逐个构造阶段目录的绝对路径。
phase_paths = [ROOT / directory for directory in phase_directories]

# 打印每个目录是否存在，先确认工程骨架没有缺失。
for path in phase_paths:
    # 输出目录名和布尔结果，帮助定位环境问题。
    print(path.name, "exists=", path.is_dir(), "path=", path)

# 只有四个阶段都存在，后续 Notebook 才有意义。
assert all(path.is_dir() for path in phase_paths)

# 输出环境检查通过的结果。
print("项目骨架检查通过。")

phase1_doc_parser exists= True path= D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\phase1_doc_parser
phase2_semantic_search exists= True path= D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\phase2_semantic_search
phase3_optimization_eval exists= True path= D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\phase3_optimization_eval
phase4_mini_rag_system exists= True path= D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\phase4_mini_rag_system
项目骨架检查通过。


In [4]:
# 定义项目中用于教学的原始文档目录。
input_directory = ROOT / "phase1_doc_parser" / "examples" / "input"

# 找出原始目录中的所有文件，并按文件名排序保证输出稳定。
input_files = sorted(path for path in input_directory.iterdir() if path.is_file())

# 打印输入目录，确认数据从哪里开始进入系统。
print("输入目录:", input_directory)

# 逐个打印文件名和扩展名，建立对数据类型的直觉。
for path in input_files:
    # suffix 表示文件扩展名，例如 .md 或 .pdf。
    print({"name": path.name, "suffix": path.suffix.lower(), "bytes": path.stat().st_size})

# 确保教学数据至少包含一个文件。
assert input_files

输入目录: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\phase1_doc_parser\examples\input
{'name': 'quickstart.md', 'suffix': '.md', 'bytes': 330}
{'name': 'retrieval-notes.md', 'suffix': '.md', 'bytes': 285}


## 2. 把目录结构翻译成数据流

目录只是工程组织方式，数据流才是系统真正的逻辑。下面用 Python 字典写出一份“项目合同”：每个阶段明确输入、处理和输出。字典不是生产配置，而是帮助你把抽象概念落到字段上。

In [5]:
# 为每个阶段记录输入、核心处理和交付物。
project_contract = {
    "phase1": {"input": "PDF/Markdown/TXT", "process": "parse + split", "output": "chunks.json"},
    "phase2": {"input": "chunks.json", "process": "BM25/Dense/RRF", "output": "rankings + qrels metrics"},
    "phase3": {"input": "rankings + qrels", "process": "benchmark + error analysis", "output": "experiment records"},
    "phase4": {"input": "validated knowledge base", "process": "FastAPI orchestration", "output": "search/chat citations"},
}

# 逐行打印合同，让每个阶段的边界可见。
for phase_name, contract in project_contract.items():
    # 输出阶段名和它的输入、处理、输出。
    print(phase_name, "->", contract)

# 确认每个阶段都写明了三类关键信息。
assert all({"input", "process", "output"} <= contract.keys() for contract in project_contract.values())

phase1 -> {'input': 'PDF/Markdown/TXT', 'process': 'parse + split', 'output': 'chunks.json'}
phase2 -> {'input': 'chunks.json', 'process': 'BM25/Dense/RRF', 'output': 'rankings + qrels metrics'}
phase3 -> {'input': 'rankings + qrels', 'process': 'benchmark + error analysis', 'output': 'experiment records'}
phase4 -> {'input': 'validated knowledge base', 'process': 'FastAPI orchestration', 'output': 'search/chat citations'}


## 3. 第一次导入：从项目代码得到一个真实结果

现在只导入 Phase 1 的解析函数，不实现细节。这里的目的不是跳过学习，而是建立一个基线：之后我们会在细分 Notebook 中手写最小版本，再和这个已经测试过的生产模块对照。

In [6]:
# 从 Phase 1 模块导入已经测试过的文件解析入口。
from phase1_doc_parser.parser import parse_file

# 选择第一个真实教学文件作为观察对象。
first_file = input_files[0]

# 调用解析函数，把一个文件统一转换为 ParsedDocument 列表。
parsed_documents = parse_file(first_file)

# 打印解析数量，确认函数确实产生了结构化结果。
print("解析结果数量:", len(parsed_documents))

# 查看第一份结果的字段，建立 Document 数据结构的直觉。
print(parsed_documents[0])

# 确保解析结果不为空，避免拿空数据继续学习。
assert parsed_documents

解析结果数量: 1
ParsedDocument(text='# Phase 1 Quickstart\n\n文档解析的目标不是尽快删除格式信息，而是保留足够的来源元数据，让后续检索结果可以回溯到原文。\n\n## Chunk 策略\n\n先按段落和换行切分，再按中文标点递归降级。overlap 用于保留跨边界的上下文，但会增加索引体积和重复召回。', source='D:\\code\\codeByCursor\\AI_EXAM\\ai-search-rag-internship\\phase1_doc_parser\\examples\\input\\quickstart.md', page=None, metadata={'format': 'markdown', 'headings': ['Phase 1 Quickstart', 'Chunk 策略']})


## 4. 保存学习起点

记录环境和项目合同，是为了让后续实验知道自己从什么状态开始。之后的报告会继续保存参数、数据规模和指标；没有这些信息，数字不能复现。

In [7]:
# 创建项目处理数据目录；目录已经存在时不会报错。
orientation_directory = ROOT / "data" / "processed"
orientation_directory.mkdir(parents=True, exist_ok=True)

# 组合本课需要保存的起点信息。
orientation_record = {
    "project_root": str(ROOT),
    "input_files": [path.name for path in input_files],
    "contract": project_contract,
    "next_notebook": "phase1/01_files_and_documents.ipynb",
}

# 选择一个稳定的 JSON 文件作为本课交付物。
orientation_path = orientation_directory / "project_orientation.json"

# 以 UTF-8 写入中文，并保留缩进方便人工阅读。
orientation_path.write_text(json.dumps(orientation_record, ensure_ascii=False, indent=2), encoding="utf-8")

# 打印交付物路径，方便在 Jupyter 文件浏览器中找到它。
print("已生成:", orientation_path)

已生成: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\data\processed\project_orientation.json


## Phase 0 验收

- [ ] 能说出四个阶段的输入和输出。
- [ ] 能解释为什么 Phase 2 不能绕过 `chunks.json` 直接读取原始文件。
- [ ] 能指出项目根目录、原始数据目录和处理数据目录。
- [ ] 已生成 `project_orientation.json`。

下一课从最基础的文件读写开始，手动构造一条 Document 记录。

## Boss Challenge：把案件的服务对象和‘必须说不知道’条件改成你自己的真实主题，并重新生成档案。

下面是**故意保持注释状态**的跟敲模板。请先自己写，再取消注释逐行运行；不要把它当成需要复制的答案。

In [8]:
# 第 1 行：先写出本挑战需要的新变量或新输入。
# challenge_input = ...

# 第 2 行：调用本课已经学会的函数或模块。
# challenge_result = ...

# 第 3 行：打印一个中间结果，先观察再下结论。
# print(challenge_result)

# 第 4 行：写一个断言，把你的理解变成机器可检查的条件。
# assert ...

## 作品检查站

作品不是‘我运行过代码’，而是别人可以在文件浏览器中找到、下一阶段可以读取、你能解释生成过程的证据。下面的检查只报告事实，不替你假装通关。

In [9]:
# 列出本关应该产生的作品路径。
quest_artifact_candidates = ['data/processed/project_orientation.json', 'data/processed/evidence_quest_profile.json']

# 把相对路径转换为项目根目录下的绝对路径。
quest_artifact_paths = [ROOT / path for path in quest_artifact_candidates]

# 只保留已经真正写入磁盘的作品。
quest_existing_artifacts = [str(path.relative_to(ROOT)) for path in quest_artifact_paths if path.is_file()]

# 保存一个不依赖外部服务的本关检查结果，方便复盘。
quest_checkpoint = {"stage": QUEST_STAGE, "existing_artifacts": quest_existing_artifacts}

# 打印检查结果，让学习者知道下一步是继续学习还是补交作品。
print("本关作品:", quest_existing_artifacts if quest_existing_artifacts else "还没有生成，请回到交付单元格")

本关作品: ['data\\processed\\project_orientation.json', 'data\\processed\\evidence_quest_profile.json']
